In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import types as T

In [2]:
spark = (
    SparkSession.builder.appName('tradeline_features')
    .config('spark.driver.memory', '30g')
    .config('spark.sql.legacy.timeParserPolicy', 'LEGACY')
    .config('spark.sql.codegen.wholeStage', 'false')
    .getOrCreate()
)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("parquet.enable.dictionary", "false")
spark.conf.set("spark.default.parallelism", 500)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 12:10:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/29 12:10:04 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/29 12:10:04 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/29 12:10:04 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [6]:
ref = spark.read.parquet(path + "ref_file.parquet")

In [7]:
df_tar = spark.read.parquet(path + "target_file.parquet")

In [8]:
case_study_users = spark.read.csv("case_study_submission_users.csv", header=True, inferSchema=True)

In [9]:
ref.show(2)

+-----------+----------+
|    user_id|retro_date|
+-----------+----------+
|68719487934|2025-03-08|
|60129552606|2025-03-08|
+-----------+----------+
only showing top 2 rows



In [10]:
ref.printSchema()

root
 |-- user_id: long (nullable = true)
 |-- retro_date: date (nullable = true)



In [11]:
df_tar.printSchema()

root
 |-- ACCOUNTTYPE: string (nullable = true)
 |-- DATEREPORTED: string (nullable = true)
 |-- ACCOUNTSTATUS: string (nullable = true)
 |-- ASSETCLASSIFICATION: string (nullable = true)
 |-- ACCOUNTSTATUS_HISTORY: string (nullable = true)
 |-- DATEOPENED: string (nullable = true)
 |-- ASSETCLASS_HISTORY: string (nullable = true)
 |-- DATECLOSED: string (nullable = true)
 |-- user_id: long (nullable = true)



In [12]:
df_tar.show(4)

+--------------------+------------+---------------+-------------------+---------------------+----------+--------------------+----------+-----------+
|         ACCOUNTTYPE|DATEREPORTED|  ACCOUNTSTATUS|ASSETCLASSIFICATION|ACCOUNTSTATUS_HISTORY|DATEOPENED|  ASSETCLASS_HISTORY|DATECLOSED|    user_id|
+--------------------+------------+---------------+-------------------+---------------------+----------+--------------------+----------+-----------+
|       Personal Loan|  29-02-2024| Closed Account|               NULL| 00000YXXXXXXXXXXX...|13-08-2023|DDDDDDXXXXXXXXXXX...|07-02-2024|34359742967|
|       Personal Loan|  29-02-2024| Closed Account|               NULL| 00000YXXXXXXXXXXX...|13-08-2023|DDDDDDXXXXXXXXXXX...|07-02-2024|34359742966|
|Short Term Person...|  15-11-2025|Current Account|           Standard| 000000YXXXXXXXXXX...|01-04-2025|1111111XXXXXXXXXX...|      NULL|34359742967|
|Short Term Person...|  15-11-2025|Current Account|           Standard| 000000YXXXXXXXXXX...|01-04-2025|11

In [13]:
df_tar.select("ACCOUNTSTATUS_HISTORY").show(2, truncate=False)

+------------------------------------------------+
|ACCOUNTSTATUS_HISTORY                           |
+------------------------------------------------+
|00000YXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|
|00000YXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|
+------------------------------------------------+
only showing top 2 rows



In [14]:
ref = ref.withColumn(
    "retro_date",
    F.to_date(F.col("retro_date"), "yyyy-MM-dd")
)

In [15]:
date_cols = ["DATEREPORTED", "DATEOPENED", "DATECLOSED"]

for col_name in date_cols:
    df_tar = df_tar.withColumn(
        col_name,
        F.to_date(F.col(col_name), "dd-MM-yyyy")
    )

In [16]:
ref.printSchema()

root
 |-- user_id: long (nullable = true)
 |-- retro_date: date (nullable = true)



In [17]:
df_tar.printSchema()

root
 |-- ACCOUNTTYPE: string (nullable = true)
 |-- DATEREPORTED: date (nullable = true)
 |-- ACCOUNTSTATUS: string (nullable = true)
 |-- ASSETCLASSIFICATION: string (nullable = true)
 |-- ACCOUNTSTATUS_HISTORY: string (nullable = true)
 |-- DATEOPENED: date (nullable = true)
 |-- ASSETCLASS_HISTORY: string (nullable = true)
 |-- DATECLOSED: date (nullable = true)
 |-- user_id: long (nullable = true)



In [18]:
df_tar.count()

5769385

In [19]:
df_tar.distinct().count()

5058420

In [20]:
ref.count()

165163

In [21]:
case_study_users.count()

90414

In [22]:
df_tar=df_tar.dropDuplicates()

In [23]:
df_tar.count()

5058420

In [24]:
# filter df_tar dates
for col_name in ["DATEREPORTED", "DATEOPENED", "DATECLOSED"]:
    df_tar = df_tar.withColumn(
        col_name,
        F.when(F.col(col_name) <= F.lit("1950-01-01").cast("date"), None)
         .otherwise(F.col(col_name))
    )

In [25]:
# filter ref.retro_date
ref = ref.withColumn(
    "retro_date",
    F.when(F.col("retro_date") < F.lit("1950-01-01").cast("date"), None)
     .otherwise(F.col("retro_date"))
)

In [26]:
df_tar.count()

5058420

In [27]:
# Taking ONLY case_study_users

target = (
    case_study_users
    .join(df_tar, on="user_id", how="left")
    .join(ref, on="user_id", how="left")
)

In [28]:
target.count()

3619252

In [29]:
# To see all distinct entries of 

In [30]:
# target.select("ACCOUNTTYPE").distinct().show(truncate=False)

In [31]:
distinct_acct = target.select("ACCOUNTTYPE").distinct()

distinct_acct.show(distinct_acct.count(), truncate=False)

+----------------------------------------------------------------------+
|ACCOUNTTYPE                                                           |
+----------------------------------------------------------------------+
|Business Loan - Priority Sector- Others                               |
|Loan Against Shares/Securities                                        |
|Credit Card                                                           |
|Business Loan - Priority Sector- Small Business                       |
|Business Non-Funded Credit Facility                                   |
|Priority Sector- Gold Loan [Secured]                                  |
|Business Loan                                                         |
|Kisan Credit Card                                                     |
|Personal Loan                                                         |
|Pradhan Mantri Awas Yojana - Credit Link Subsidy Scheme MAY CLSS      |
|Loan on Credit Card                               

In [32]:
distinct_acct.count()

51

In [31]:
mapping = spark.read.csv("mapping.csv", header=True, inferSchema=True)

In [34]:
mapping.show(2)

+--------------------+------------+
|         ACCOUNTTYPE|PRODUCT_DESC|
+--------------------+------------+
| Loan on Credit Card|          PL|
|Loan to Professional|          PL|
+--------------------+------------+
only showing top 2 rows



In [16]:
mapping.select("ACCOUNTTYPE").distinct().count()

23

In [17]:
mapping.count()

52

In [17]:
# Add PRODUCT_DESC column to target

In [32]:
target1 = target.join(
    F.broadcast(mapping),
    on="ACCOUNTTYPE",
    how="left"
)

In [39]:
target1.count()

3619252

In [40]:
target1.show(3)

+--------------------+-------+------------+--------------------+-------------------+---------------------+----------+--------------------+----------+----------+------------+
|         ACCOUNTTYPE|user_id|DATEREPORTED|       ACCOUNTSTATUS|ASSETCLASSIFICATION|ACCOUNTSTATUS_HISTORY|DATEOPENED|  ASSETCLASS_HISTORY|DATECLOSED|retro_date|PRODUCT_DESC|
+--------------------+-------+------------+--------------------+-------------------+---------------------+----------+--------------------+----------+----------+------------+
|         Credit Card|   9993|  2025-11-15|Charge Off/Writte...|               NULL| W43221100000000YX...|2024-07-11|DDDDDDDDDDDDDDDDX...|      NULL|2025-04-27|        NULL|
|Short Term Person...|   9993|  2024-08-31|      Closed Account|           Standard| 0000YXXXXXXXXXXXX...|2024-03-27|11111XXXXXXXXXXXX...|2024-08-27|2025-04-27|          PL|
|         Credit Card|   9993|  2025-11-15|90-119 days past due|               NULL| 4333210000000XXXX...|2024-09-13|DDDDDDDDDDDDD

In [41]:
# All filters
# Keep only PL, BL

In [33]:
target2 = target1.filter(F.col("DATEOPENED").isNotNull())\
    .filter(F.col("DATEREPORTED").isNotNull())\
    .filter(F.col("retro_date").isNotNull())\
    .filter(F.trunc(F.col("DATEREPORTED"), "month") > F.trunc(F.col("retro_date"), "month"))\
    .filter(F.col("DATEOPENED") <= F.col("retro_date"))\
    .filter((F.col("DATECLOSED").isNull()) | (F.col("DATECLOSED") > F.col("retro_date")))\
    .filter((F.col("DATECLOSED").isNull()) | (F.col("DATEOPENED") < F.col("DATECLOSED")))\
    .filter(F.col("PRODUCT_DESC").isin("PL", "BL"))

In [35]:
target2.count()

383980

# DPD

In [36]:
target2.select(
    F.length("ACCOUNTSTATUS_HISTORY").alias("ACCOUNTSTATUS_HISTORY_len"),
    F.length("ASSETCLASS_HISTORY").alias("ASSETCLASS_HISTORY_len")
).distinct().orderBy("ACCOUNTSTATUS_HISTORY_len", "ASSETCLASS_HISTORY_len").show(truncate=False)

+-------------------------+----------------------+
|ACCOUNTSTATUS_HISTORY_len|ASSETCLASS_HISTORY_len|
+-------------------------+----------------------+
|48                       |48                    |
+-------------------------+----------------------+



In [37]:
# So, 48 months data from DATEREPORTED. No null.

In [38]:
target2 = target2.withColumn(
    "month_diff",
    F.months_between(F.trunc(F.col("DATEREPORTED"), "month"), F.trunc(F.col("retro_date"), "month")).cast("int")
)

In [39]:
target2 = target2.filter(F.col("month_diff") > 0)

In [47]:
target2.show(2)

+--------------------+-----------+------------+------------------+-------------------+---------------------+----------+--------------------+----------+----------+------------+----------+
|         ACCOUNTTYPE|    user_id|DATEREPORTED|     ACCOUNTSTATUS|ASSETCLASSIFICATION|ACCOUNTSTATUS_HISTORY|DATEOPENED|  ASSETCLASS_HISTORY|DATECLOSED|retro_date|PRODUCT_DESC|month_diff|
+--------------------+-----------+------------+------------------+-------------------+---------------------+----------+--------------------+----------+----------+------------+----------+
|Short Term Person...|      13608|  2025-07-15|    Closed Account|                   | 00YXXXXXXXXXXXXXX...|2025-04-08|DDDXXXXXXXXXXXXXX...|2025-07-13|2025-04-08|          PL|         3|
|       Personal Loan|42949675966|  2025-09-15|1-29 days past due|           Standard| 000000000YXXXXXXX...|2024-11-09|1111111111XXXXXXX...|      NULL|2025-03-08|          PL|         6|
+--------------------+-----------+------------+------------------

In [48]:
target2.select(F.max("month_diff").alias("max_month_diff"), F.min("month_diff").alias("min_month_diff")).show()

+--------------+--------------+
|max_month_diff|min_month_diff|
+--------------+--------------+
|            12|             1|
+--------------+--------------+



In [49]:
# target2 = target2.filter(F.col("month_diff") > 0) . Alredy applied .filter(F.trunc(F.col("DATEREPORTED"), "month") > F.trunc(F.col("retro_date"), "month"))

In [50]:
target2.printSchema()

root
 |-- ACCOUNTTYPE: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- DATEREPORTED: date (nullable = true)
 |-- ACCOUNTSTATUS: string (nullable = true)
 |-- ASSETCLASSIFICATION: string (nullable = true)
 |-- ACCOUNTSTATUS_HISTORY: string (nullable = true)
 |-- DATEOPENED: date (nullable = true)
 |-- ASSETCLASS_HISTORY: string (nullable = true)
 |-- DATECLOSED: date (nullable = true)
 |-- retro_date: date (nullable = true)
 |-- PRODUCT_DESC: string (nullable = true)
 |-- month_diff: integer (nullable = true)



In [51]:
target2.show(1)

+--------------------+-------+------------+--------------+-------------------+---------------------+----------+--------------------+----------+----------+------------+----------+
|         ACCOUNTTYPE|user_id|DATEREPORTED| ACCOUNTSTATUS|ASSETCLASSIFICATION|ACCOUNTSTATUS_HISTORY|DATEOPENED|  ASSETCLASS_HISTORY|DATECLOSED|retro_date|PRODUCT_DESC|month_diff|
+--------------------+-------+------------+--------------+-------------------+---------------------+----------+--------------------+----------+----------+------------+----------+
|Short Term Person...|  13608|  2025-07-15|Closed Account|                   | 00YXXXXXXXXXXXXXX...|2025-04-08|DDDXXXXXXXXXXXXXX...|2025-07-13|2025-04-08|          PL|         3|
+--------------------+-------+------------+--------------+-------------------+---------------------+----------+--------------------+----------+----------+------------+----------+
only showing top 1 row



In [40]:
dpd_cols = []

for i in range(9):
    
    pos = F.col("month_diff") + F.lit(1 - i)

    status = F.when(
        pos >= 1,
        F.expr(
            f"substring(ACCOUNTSTATUS_HISTORY, month_diff + {1-i}, 1)"
        )
    )

    
    asset = F.when(
        pos >= 1,
        F.expr(
            f"substring(ASSETCLASS_HISTORY, month_diff + {1-i}, 1)"
        )
    )

    status_dpd = (
        F.when(status.isin("0", "K"), 0)
         .when(status == "1", 1)
         .when(status == "2", 30)
         .when(status.isin("3", "M"), 60)
         .when(status.isin("4", "Q"), 90)
         .when(status == "5", 120)
         .when(status == "R", 150)
         .when(status.isin("6", "A", "G", "H", "N"), 180)
         .when(status.isin("7", "8", "B"), 360)
         .when(status == "C", 540)
         .when(status == "F", 720)
         .when(status == "W", 899)
    )

    class_label = (
        F.when(asset == "1", F.lit("STD"))
         .when(asset == "2", F.lit("SUB"))
         .when(asset == "3", F.lit("DBT"))
         .when(asset == "4", F.lit("Loss"))
         .when(asset == "5", F.lit("SMA"))
         .when(asset == "X", F.lit("XXX"))
         .when(asset == "0", F.lit("0"))
    )
 


    class_bump = (
        F.when(class_label == "SUB", 90)
         .when(class_label == "DBT", 180)
         .when(class_label == "Loss", 180)
         .when(class_label == "SMA", 60)
    )

    raw_dpd = (
        F.when(
            status_dpd.isNull() & class_bump.isNull(),
            None
        )

        .when(
            status_dpd.isNull(),
            class_bump
        )

        .when(
            class_bump.isNull(),
            status_dpd
        )

        .otherwise(
            F.greatest(status_dpd, class_bump)
        )
    )
    

    final_dpd = (
        F.when(raw_dpd.isNull(), None)
         .when(raw_dpd >= 360, 360)
         .when(raw_dpd >= 180, 180)
         .when(raw_dpd >= 150, 150)
         .when(raw_dpd >= 120, 120)
         .when(raw_dpd >= 90, 90)
         .when(raw_dpd >= 60, 60)
         .when(raw_dpd >= 30, 30)
         .when(raw_dpd > 0, 1)
         .otherwise(0)
    )

    
    col_name = f"dpd_{i}"

   
    target2 = target2.withColumn(
        col_name,
        final_dpd
    )

    dpd_cols.append(col_name)

In [41]:
target2 = target2.withColumn(

    "valid_dpd_months",

    sum(

        F.when(F.col(c).isNotNull(), 1).otherwise(0)

        for c in dpd_cols

    )

)

target2 = target2.withColumn(

    "ever60_trade",

    F.when(

        (F.col("valid_dpd_months") >= 3) &

        (

            F.greatest(

                *[F.col(c) for c in dpd_cols]

            ) >= 60

        ),

        1

    ).otherwise(0)

)

In [42]:
user_target = (
    target2
    .groupBy("user_id")
    .agg(
        F.max("ever60_trade").alias("user_ever60_9m")
    )
)

In [43]:
user_target.count()

90414

In [44]:
user_target.coalesce(1).write.mode("overwrite").parquet(
    "user_target.parquet"
)

26/06/29 12:14:15 WARN DAGScheduler: Broadcasting large task binary with size 5.3 MiB
26/06/29 12:14:34 WARN DAGScheduler: Broadcasting large task binary with size 9.9 MiB
